# eRayz COD MEGA — sunxds_0.7.8 + ALLE CoD Datasets
---
Kombiniert 5 CoD-Datasets (~9.000+ Bilder) mit sunxds_0.7.8 (30.000 FPS).

**Datasets:**
- COD Complete (4.643)
- COD MW Warzone (enemy + head)
- callofduty (2.248 enemy + team)
- cod (1.579 player)
- Warzone YOLOv8 (751 enemy)

**ANLEITUNG:**
1. Laufzeit > GPU (T4)
2. Alle Zellen ausfuehren
3. sunxds_0.7.8.pt hochladen wenn gefragt
4. Roboflow API Key eintragen
5. ~40 Min warten
6. erayz_cod_mega.onnx wird heruntergeladen

In [ ]:
# SCHRITT 1: Installieren
!pip install -q ultralytics roboflow pyyaml
print('OK')

In [ ]:
# SCHRITT 2: GPU Check
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('KEINE GPU! Laufzeit > Laufzeittyp > GPU (T4)')

In [ ]:
# SCHRITT 3: sunxds_0.7.8.pt hochladen
import os
from google.colab import files

if not os.path.exists('sunxds_0.7.8.pt'):
    print('Lade sunxds_0.7.8.pt hoch...')
    uploaded = files.upload()
else:
    print('sunxds_0.7.8.pt vorhanden!')

size = os.path.getsize('sunxds_0.7.8.pt') / (1024*1024)
print(f'Modell: {size:.1f} MB')

In [ ]:
# SCHRITT 4: ALLE CoD Datasets herunterladen
from roboflow import Roboflow
import shutil, glob, yaml

# ═══════════════════════════════════════
RF_API_KEY = 'DEIN_API_KEY'  # <-- HIER
# ═══════════════════════════════════════

rf = Roboflow(api_key=RF_API_KEY)

# Mega-Ordner erstellen
os.makedirs('mega_dataset/train/images', exist_ok=True)
os.makedirs('mega_dataset/train/labels', exist_ok=True)
os.makedirs('mega_dataset/valid/images', exist_ok=True)
os.makedirs('mega_dataset/valid/labels', exist_ok=True)

total = 0

# Dataset-Liste: (workspace, project, version)
datasets = [
    ('aimbots-gm16b', 'cod-complet-ubmf3', [1, 2, 3]),
    ('kolly-ku5ew', 'cod-mw-warzone', [1, 2, 3]),
    ('call-of-duty-lpxft', 'callofduty', [1, 2]),
    ('cod-auguc', 'cod-lbdrt', [1]),
    ('warzone-n5a13', 'warzone-yolov8', [1, 2]),
]

downloaded = []

for ws, proj, versions in datasets:
    success = False
    for v in versions:
        try:
            print(f'\nLade: {proj} v{v}...')
            project = rf.workspace(ws).project(proj)
            ds = project.version(v).download('yolov8')
            downloaded.append((proj, ds.location))
            success = True
            break
        except Exception as e:
            print(f'  v{v} fehlgeschlagen: {e}')
            continue
    if not success:
        print(f'  WARNUNG: {proj} konnte nicht geladen werden — uebersprungen')

print(f'\n{len(downloaded)} Datasets heruntergeladen!')
for name, loc in downloaded:
    print(f'  - {name}: {loc}')

In [ ]:
# SCHRITT 5: Alle Datasets zusammenfuehren
# Alle Labels auf eine einzige Klasse (0 = player) normalisieren
import re

img_count = 0

for proj_name, ds_loc in downloaded:
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(ds_loc, split, 'images')
        lbl_dir = os.path.join(ds_loc, split, 'labels')
        
        if not os.path.exists(img_dir):
            continue
        
        # Ziel: train oder valid
        target = 'valid' if split == 'valid' else 'train'
        
        imgs = glob.glob(os.path.join(img_dir, '*'))
        for img_path in imgs:
            fname = os.path.basename(img_path)
            # Prefix um Duplikate zu vermeiden
            new_fname = f'{proj_name}_{fname}'
            
            # Bild kopieren
            dst_img = os.path.join('mega_dataset', target, 'images', new_fname)
            shutil.copy2(img_path, dst_img)
            
            # Label kopieren + alle Klassen auf 0 (player) setzen
            lbl_name = os.path.splitext(fname)[0] + '.txt'
            lbl_path = os.path.join(lbl_dir, lbl_name)
            new_lbl = os.path.splitext(new_fname)[0] + '.txt'
            dst_lbl = os.path.join('mega_dataset', target, 'labels', new_lbl)
            
            if os.path.exists(lbl_path):
                with open(lbl_path) as f:
                    lines = f.readlines()
                # Alle Klassen-IDs auf 0 (player) setzen
                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        parts[0] = '0'  # Alles = player
                        new_lines.append(' '.join(parts) + '\n')
                with open(dst_lbl, 'w') as f:
                    f.writelines(new_lines)
            
            img_count += 1

train_count = len(glob.glob('mega_dataset/train/images/*'))
val_count = len(glob.glob('mega_dataset/valid/images/*'))
print(f'\nMEGA DATASET FERTIG!')
print(f'  Training: {train_count} Bilder')
print(f'  Validation: {val_count} Bilder')
print(f'  TOTAL: {train_count + val_count} Bilder')
print(f'  Klasse: player (alles vereint)')

In [ ]:
# SCHRITT 6: data.yaml erstellen
import yaml

data_yaml = {
    'path': os.path.abspath('mega_dataset'),
    'train': 'train/images',
    'val': 'valid/images',
    'nc': 1,
    'names': {0: 'player'}
}

yaml_path = 'mega_dataset/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print('data.yaml erstellt:')
print(f'  Klassen: 1 (player)')
print(f'  Train: {data_yaml["train"]}')
print(f'  Val: {data_yaml["val"]}')

In [ ]:
# SCHRITT 7: MEGA FINE-TUNING
from ultralytics import YOLO

model = YOLO('sunxds_0.7.8.pt')
print(f'Basis: sunxds_0.7.8 ({len(model.names)} Klassen)')
print(f'Fine-Tuning auf {train_count + val_count} CoD Bilder...')
print(f'Das dauert ca. 35-45 Minuten.\n')

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=12,
    name='erayz_cod_mega',
    # Fine-Tuning Settings
    lr0=0.0006,
    lrf=0.01,
    warmup_epochs=3,
    freeze=8,
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.8,
    mixup=0.1,
    translate=0.1,
    scale=0.4,
    degrees=3.0,
)
print('\nTRAINING FERTIG!')

In [ ]:
# SCHRITT 8: Ergebnisse
from IPython.display import Image, display

train_dir = 'runs/detect/erayz_cod_mega'
if not os.path.exists(train_dir):
    for d in sorted(os.listdir('runs/detect')):
        if 'mega' in d:
            train_dir = f'runs/detect/{d}'

print(f'Ergebnisse: {train_dir}')
if os.path.exists(f'{train_dir}/results.png'):
    display(Image(filename=f'{train_dir}/results.png', width=800))
if os.path.exists(f'{train_dir}/val_batch0_pred.jpg'):
    display(Image(filename=f'{train_dir}/val_batch0_pred.jpg', width=800))

In [ ]:
# SCHRITT 9: ONNX Export
import shutil

best = YOLO(f'{train_dir}/weights/best.pt')
best.export(format='onnx', imgsz=640, simplify=True, dynamic=False)

shutil.copy(f'{train_dir}/weights/best.onnx', 'erayz_cod_mega.onnx')
shutil.copy(f'{train_dir}/weights/best.pt', 'erayz_cod_mega.pt')

size = os.path.getsize('erayz_cod_mega.onnx') / (1024*1024)
print(f'\n{"="*50}')
print(f'FERTIG: erayz_cod_mega.onnx ({size:.1f} MB)')
print(f'{"="*50}')
print(f'\nDieses Modell kombiniert:')
print(f'  sunxds_0.7.8 (30.000 FPS) +')
print(f'  {train_count + val_count} CoD-spezifische Bilder')
print(f'\n  = MEGA COD MODELL')
print(f'\nLege erayz_cod_mega.onnx in Downloads/backend/')

In [ ]:
# SCHRITT 10: Download
from google.colab import files
files.download('erayz_cod_mega.onnx')
files.download('erayz_cod_mega.pt')
print('Download laeuft!')
print('  .onnx = fuer Aimbot')
print('  .pt = fuer spaeteres Weitertraining')